In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import root_mean_squared_error

import torch
from transformers import pipeline, CLIPModel, CLIPProcessor
from PIL import Image
import matplotlib.pyplot as plt

### Dataset generation

In [3]:
n_vectors = 1000
vector_len = 8
n_blocks = 4

block_len = vector_len // n_blocks
# block_len = math.ceil(vector_len / n_blocks)
# remainder = vector_len - vector_len // n_blocks * n_blocks   # if vector_len !% n_blocks

noise_level = 0.3

In [4]:
seed = 42
np.random.seed(seed)

cluster_centers = np.empty((n_blocks, block_len))
for cluster in range(n_blocks):
    cluster_centers[cluster] = np.random.uniform(-1, 1, block_len) + cluster*2

vectors_dataset = np.zeros((n_vectors, vector_len))

for i in range(n_vectors):
    for block in range(n_blocks):
        vectors_dataset[i][(block_len * block) : (block_len * (block+1))] = cluster_centers[block] + np.random.normal(0, noise_level, block_len)

vectors_dataset[0:3]

array([[ 0.22284408,  1.13165903,  2.32314557,  2.36008498,  3.17301197,
         3.17227011,  5.18875591,  6.15836822],
       [-0.76839511,  0.73274235,  2.16013855,  2.29159117,  3.03963006,
         2.88829793,  5.55586186,  6.6646194 ],
       [-0.2306613 ,  0.47400416,  2.30067307,  2.23059375,  2.96673921,
         3.42469845,  4.93597562,  6.64484417]])

### Vanilla product quantization (with K-Means clustering)

In [16]:
def product_quantize(vectors, 
                     n_blocks: int = 4, 
                     n_clusters: int = 1,
                     seed: int = 42, 
                     cluster_init: int = 3):
    block_len = vectors.shape[1] // n_blocks
    subspaces = []
    cluster_labels = []
    cluster_centroids = []
    subspaces_q = []

    for i in range(0, vectors.shape[1] - block_len + 1, block_len):
        subspaces.append(vectors[:, i:(i + block_len)])

    for block in range(n_blocks):
        kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=cluster_init)
        cluster_labels.append(kmeans.fit_predict(subspaces[block]))
        cluster_centroids.append(kmeans.cluster_centers_)

    for i in range(n_blocks):
        subspaces_q.append( cluster_centroids[i][cluster_labels[i]] )
    
    return np.concat(subspaces_q, axis = 1)   


In [22]:
vectors_quantized = product_quantize(vectors_dataset, n_blocks, 2, seed, cluster_init=3)
print("Product quantization RMSE for randomly generated vectors: ", root_mean_squared_error(vectors_dataset, vectors_quantized))

Product quantization RMSE for randomly generated vectors:  0.24553920033432028


### Embedding extraction

In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip = clip.eval()

In [58]:
def get_image_embeddings(model, images_paths: list, device: torch.device):
    images = [Image.open(image_path) for image_path in images_paths]
    inputs = processor(images=images, return_tensors="pt").to(device)
    
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
        image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    
    return image_embeddings.cpu().numpy().astype(np.float32)

In [59]:
images_folder = "./images/"
images_names = ["cat.jpg", "dog.jpg", "wolf.png", "gosling.png"]

In [61]:
image_embeddings = get_image_embeddings(
    clip, 
    [images_folder + image_name for image_name in images_names], 
    device
) 
print(f"Embeddings of images {images_names} have shape:  {image_embeddings.shape}")

Embeddings of images ['cat.jpg', 'dog.jpg', 'wolf.png', 'gosling.png'] have shape:  (4, 512)


In [62]:
image_embeddings[0][:10]

array([-0.00465205, -0.01560188,  0.01688051,  0.02128985,  0.01414021,
       -0.02529875, -0.01026839,  0.04886029,  0.00564997,  0.00126772],
      dtype=float32)

### Vanilla product quantization application to embeddings

In [63]:
embeddings_quantized = product_quantize(image_embeddings, n_blocks, 1, seed, cluster_init=3)
print("Product quantization RMSE for randomly generated vectors: ", root_mean_squared_error(image_embeddings, embeddings_quantized))

Product quantization RMSE for randomly generated vectors:  0.023134570568799973
